In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile

zip_path = "/content/drive/MyDrive/archive (3).zip"
extract_path = "/content/f1_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("F1 dataset extracted successfully!")

F1 dataset extracted successfully!


In [4]:
import os
os.listdir("/content/f1_data")

['circuits.csv',
 'qualifying.csv',
 'constructor_results.csv',
 'seasons.csv',
 'drivers.csv',
 'sprint_results.csv',
 'constructor_standings.csv',
 'races.csv',
 'pit_stops.csv',
 'results.csv',
 'constructors.csv',
 'status.csv',
 'lap_times.csv',
 'driver_standings.csv']

In [14]:
# 🏎️ F1 Predictor Pro - FIXED FULL CODE
# Works with your dataset in Colab

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# ---------------- LOAD DATA ----------------
drivers = pd.read_csv("/content/f1_data/drivers.csv")
results = pd.read_csv("/content/f1_data/results.csv")
races = pd.read_csv("/content/f1_data/races.csv")
constructors = pd.read_csv("/content/f1_data/constructors.csv")
circuits = pd.read_csv("/content/f1_data/circuits.csv")

# ---------------- CLEAN DATA ----------------
results["positionOrder"] = pd.to_numeric(results["positionOrder"], errors="coerce")
results["grid"] = pd.to_numeric(results["grid"], errors="coerce")

# Driver full name
drivers["driverName"] = drivers["forename"] + " " + drivers["surname"]

# Rename columns before merge
constructors = constructors.rename(columns={"name": "teamName"})
circuits = circuits.rename(columns={"name": "circuitName"})
races = races.rename(columns={"name": "raceName"})

# ---------------- MERGE ----------------
df = results.merge(drivers[["driverId", "driverName"]], on="driverId", how="left")
df = df.merge(races[["raceId", "year", "raceName", "circuitId"]], on="raceId", how="left")
df = df.merge(constructors[["constructorId", "teamName"]], on="constructorId", how="left")
df = df.merge(circuits[["circuitId", "circuitName"]], on="circuitId", how="left")

# Remove missing rows
df = df.dropna(subset=["driverName", "teamName", "circuitName", "positionOrder", "grid"])

# Use modern era only
df = df[df["year"] >= 2015]

# ---------------- ENCODE ----------------
driver_codes = {name:i for i, name in enumerate(sorted(df["driverName"].unique()))}
team_codes = {name:i for i, name in enumerate(sorted(df["teamName"].unique()))}
circuit_codes = {name:i for i, name in enumerate(sorted(df["circuitName"].unique()))}

df["driverCode"] = df["driverName"].map(driver_codes)
df["teamCode"] = df["teamName"].map(team_codes)
df["circuitCode"] = df["circuitName"].map(circuit_codes)

# ---------------- MODEL ----------------
X = df[["driverCode", "teamCode", "circuitCode", "grid", "year"]]
y = df["positionOrder"]

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X, y)

# ---------------- UI ----------------
title = widgets.HTML("""
<h1 style='color:#e10600;'>🏎️ F1 Predictor Pro</h1>
<h4>Predict Formula 1 Results using Machine Learning</h4>
""")

driver_dd = widgets.Dropdown(
    options=sorted(driver_codes.keys()),
    description="Driver:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

team_dd = widgets.Dropdown(
    options=sorted(team_codes.keys()),
    description="Team:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

circuit_dd = widgets.Dropdown(
    options=sorted(circuit_codes.keys()),
    description="Circuit:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

grid_slider = widgets.IntSlider(
    value=5, min=1, max=20,
    description="Grid:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

year_slider = widgets.IntSlider(
    value=2026, min=2015, max=2026,
    description="Year:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

button = widgets.Button(
    description="Predict Result",
    button_style="danger",
    icon="flag-checkered"
)

output = widgets.Output()

# ---------------- FUNCTION ----------------
def predict_result(b):
    with output:
        clear_output()

        driver = driver_dd.value
        team = team_dd.value
        circuit = circuit_dd.value
        grid = grid_slider.value
        year = year_slider.value

        input_data = pd.DataFrame([[
            driver_codes[driver],
            team_codes[team],
            circuit_codes[circuit],
            grid,
            year
        ]], columns=["driverCode", "teamCode", "circuitCode", "grid", "year"])

        pred = round(model.predict(input_data)[0], 2)

        if pred <= 1.5:
            status = "🥇 Strong Win Chance"
        elif pred <= 3:
            status = "🏆 Likely Podium"
        elif pred <= 10:
            status = "🎯 Points Finish"
        else:
            status = "⚠️ Tough Race"

        display(HTML(f"""
        <div style='background:#111;color:#ffffff;padding:20px;border-radius:15px; font-size:24px; font-weight:bold; line-height:1.8;'>
            <h2 style='color:#ffffff;'>{driver}</h2>
            <p>🏎️ Team: {team}</p>
            <p>📍 Circuit: {circuit}</p>
            <p>🔢 Grid Start: P{grid}</p>
            <p>📅 Season: {year}</p>
            <hr>
            <h2 style='color:#00ffcc;'>Predicted Finish: P{pred}</h2>
            <h3 style='color:#ffffff;'>{status}</h3>
        </div>
        """))

        # Driver yearly chart
        d = df[df["driverName"] == driver]
        yearly = d.groupby("year")["positionOrder"].mean()

        plt.figure(figsize=(8,4))
        plt.plot(yearly.index, yearly.values, marker="o", linewidth=2)
        plt.gca().invert_yaxis()
        plt.title(f"{driver} Average Finish by Year")
        plt.xlabel("Year")
        plt.ylabel("Average Finish")
        plt.grid(True)
        plt.show()

button.on_click(predict_result)

# ---------------- DISPLAY ----------------
display(
    title,
    driver_dd,
    team_dd,
    circuit_dd,
    grid_slider,
    year_slider,
    button,
    output
)

HTML(value="\n<h1 style='color:#e10600;'>🏎️ F1 Predictor Pro</h1>\n<h4>Predict Formula 1 Results using Machine…

Dropdown(description='Driver:', layout=Layout(width='500px'), options=('Alexander Albon', 'Alexander Rossi', '…

Dropdown(description='Team:', layout=Layout(width='500px'), options=('Alfa Romeo', 'AlphaTauri', 'Alpine F1 Te…

Dropdown(description='Circuit:', layout=Layout(width='500px'), options=('Albert Park Grand Prix Circuit', 'Aut…

IntSlider(value=5, description='Grid:', layout=Layout(width='500px'), max=20, min=1, style=SliderStyle(descrip…

IntSlider(value=2026, description='Year:', layout=Layout(width='500px'), max=2026, min=2015, style=SliderStyle…

Button(button_style='danger', description='Predict Result', icon='flag-checkered', style=ButtonStyle())

Output()